In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
_, truth_labels = next(iter(transformed_dataset))
truth_labels.shape, truth_labels

(torch.Size([5, 5]),
 tensor([[8.0000, 0.5848, 0.7321, 0.1205, 0.3393],
         [8.0000, 0.4196, 0.8482, 0.1741, 0.2902],
         [8.0000, 0.0714, 0.8259, 0.1250, 0.3482],
         [8.0000, 0.5357, 0.6562, 0.1071, 0.2812],
         [8.0000, 0.5893, 0.5402, 0.0714, 0.0893]]))

In [4]:
bboxes = truth_labels[:, -4:]
bboxes.shape, bboxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [5]:
truth_labels[:, 0]

tensor([8., 8., 8., 8., 8.])

In [6]:
bbox_preds = torch.randn(100, 21)
bbox_preds

tensor([[ 0.5615, -0.7589,  1.8373,  ...,  0.4595,  0.5164,  0.3572],
        [ 1.6454, -0.5157, -1.1682,  ..., -1.2440,  0.3895, -1.1277],
        [-0.2589, -1.1412,  0.0707,  ..., -2.7357, -0.4035,  1.9783],
        ...,
        [-0.3714, -0.3427, -0.6472,  ..., -0.5377,  1.3303, -0.1983],
        [ 0.0730, -0.6808,  0.8232,  ..., -0.3010,  0.4389, -0.1342],
        [ 0.8202, -0.3850,  0.7979,  ...,  1.3943,  0.9898, -0.2273]])

In [7]:
class_preds = torch.randn(100, 21)

In [8]:
for idx in truth_labels[:, 0]: print(int(idx))

8
8
8
8
8


In [9]:
import torch.nn.functional as F

In [10]:
from src.utilities import giou

bbox_preds = torch.randn(100, 4)

In [11]:
truth_boxes = truth_labels[..., -4:]
truth_boxes.shape

torch.Size([5, 4])

In [12]:
 truth_boxes[0], bbox_preds[0], bbox_preds[1], bbox_preds[2], bbox_preds[3], bbox_preds[4]

(tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([-1.4798,  0.7312,  1.3502,  1.9755]),
 tensor([-1.0235,  0.0500, -0.1616,  0.6508]),
 tensor([ 0.2072, -0.5859, -1.6482, -2.5220]),
 tensor([ 0.7372, -1.7380, -0.4319, -0.6199]),
 tensor([-0.9205, -0.5079, -1.0043, -0.3017]))

In [13]:
preds_clone = bbox_preds.clone().detach()

# INTERSECTION COORDINATES
preds_clone[..., 0] = torch.max(preds_clone[..., 0], truth_boxes[0][0])
preds_clone[..., 1] = torch.max(preds_clone[..., 1], truth_boxes[0][1])
preds_clone[..., 2] = torch.min(preds_clone[..., 2], truth_boxes[0][2])
preds_clone[..., 3] = torch.min(preds_clone[..., 3], truth_boxes[0][3])

preds_clone.shape, preds_clone[0], preds_clone[1], preds_clone[2], preds_clone[3], preds_clone[4]

(torch.Size([100, 4]),
 tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([ 0.5848,  0.7321, -0.1616,  0.3393]),
 tensor([ 0.5848,  0.7321, -1.6482, -2.5220]),
 tensor([ 0.7372,  0.7321, -0.4319, -0.6199]),
 tensor([ 0.5848,  0.7321, -1.0043, -0.3017]))

In [14]:
from src.utilities import intersection_coords

preds_clone_2 = bbox_preds.clone().detach()

intersection_coords(preds_clone_2, truth_boxes[0])

preds_clone_2

tensor([[ 0.5848,  0.7321,  0.1205,  0.3393],
        [ 0.5848,  0.7321, -0.1616,  0.3393],
        [ 0.5848,  0.7321, -1.6482, -2.5220],
        [ 0.7372,  0.7321, -0.4319, -0.6199],
        [ 0.5848,  0.7321, -1.0043, -0.3017],
        [ 0.5848,  1.4344,  0.1205, -1.4208],
        [ 0.5848,  0.7321, -0.0346, -1.5016],
        [ 0.5848,  0.7321,  0.1205,  0.3372],
        [ 1.0353,  0.9966, -1.4897,  0.3393],
        [ 0.5848,  0.7427,  0.1205, -0.0340],
        [ 0.5848,  0.7321,  0.1205, -0.4522],
        [ 0.5848,  1.0156, -2.2419, -0.4757],
        [ 0.5848,  0.7321,  0.1205,  0.3393],
        [ 0.5848,  0.7321, -1.1132,  0.3393],
        [ 0.5848,  0.7321, -1.0038,  0.1109],
        [ 0.5848,  0.7321, -0.3031, -0.2919],
        [ 0.6701,  0.7321, -0.3438,  0.3393],
        [ 0.5848,  0.7321,  0.1205,  0.3393],
        [ 0.5848,  0.7321, -1.2543, -0.4215],
        [ 0.5848,  0.7321,  0.0198, -0.5102],
        [ 0.5848,  0.7321,  0.1205, -0.2561],
        [ 0.5848,  0.7321, -0.3820

In [15]:
w = torch.clamp(preds_clone[..., 2] - preds_clone[..., 0], min=0)
h = torch.clamp(preds_clone[..., 3] - preds_clone[..., 1], min=0)

In [16]:
w[0], w[1], h[0], h[1], preds_clone[0], preds_clone[1]

(tensor(0.),
 tensor(0.),
 tensor(0.),
 tensor(0.),
 tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([ 0.5848,  0.7321, -0.1616,  0.3393]))

In [17]:
areas = w * h
areas.shape, areas[0], areas[1]

(torch.Size([100]), tensor(0.), tensor(0.))